# 03 - Frame Features (offline, question-independent)

The first stage of the proposed method. For every extracted video (test + calibration), and **without looking at
the question**, this notebook computes:

| Feature | How | Used by |
|---|---|---|
| `emb` | SigLIP2 image embedding (768-d, L2-normalised) | relevance to the question, near-duplicate removal, segmentation |
| `motion` | mean absolute difference of 64x64 grayscale thumbnails vs. the previous frame | action / event and counting policies |
| `sharp` | variance of the Laplacian on the 640x360-fitted grayscale frame | frame quality |
| `contrast` | grayscale standard deviation (near 0 = black / blank / fade frame) | frame quality |
| `seg` | contiguous time segment id (see below) | every policy |

It also embeds every question's text with the SigLIP2 text encoder, which gives question-to-frame relevance in
notebook 04.

### Temporal segmentation
The distance between consecutive frames is `d_t = 1 - cos(emb_t, emb_{t-1})`. A frame starts a new segment when:
- `d_t` is an outlier for that video (`> median + K_MAD * 1.4826 * MAD`);
- it is at least `MIN_SEG_S` seconds from any other boundary (strongest boundaries are accepted first);
- segments longer than `MAX_SEG_S` are also split into equal parts, so long static shots still offer several
  places to sample from.

Segments stay in time order. Two segments that look alike but happen at different times remain **separate**, which
matters for counting and temporal questions.

Output: `data/features/fps_<n>/<dataset>/<frame_key>.npz` + `data/features/questions.npz`.

In [ ]:
import os
import json
import math
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Configuration

In [ ]:
# The notebook lives in "Proposed Approach/"; data is in the project root one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
SUBSET_DIR = DATA_ROOT / "subset"
FRAMES_ROOT = DATA_ROOT / "frames"
FEATURES_ROOT = DATA_ROOT / "features"
assert (SUBSET_DIR / "manifest.jsonl").exists(), f"run notebook 01 first (looked in {SUBSET_DIR})"

FPS = {"mvbench": 5.0, "videomme": 1.0}           # must match notebook 01
SPLITS = {"test": "manifest.jsonl", "calib": "manifest_calib.jsonl"}
BOX = (640, 360)                                  # frames are fitted into this box (no upscaling) before quality features

SIGLIP_ID = "google/siglip2-base-patch16-224"
EMB_BATCH = 64
FEATURE_VERSION = 1                               # bump to force a recompute after changing the code below

SEGMENTATION = {                                  # per dataset, in seconds
    "mvbench":  dict(min_seg_s=1.0, max_seg_s=None, k_mad=3.0),
    "videomme": dict(min_seg_s=3.0, max_seg_s=60.0, k_mad=3.0),
}

FEATURES_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Videos to process

In [ ]:
def fps_dirname(fps):
    return "fps_" + f"{fps:g}".replace(".", "p")


def load_rows():
    rows = []
    for split, name in SPLITS.items():
        p = SUBSET_DIR / name
        if not p.exists():
            print(f"({split}: {name} not found, skipped)")
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                r["split"] = split
                r["frame_dir"] = str(FRAMES_ROOT / fps_dirname(FPS[r["dataset"]]) / r["dataset"] / r["frame_key"])
                r["feature_path"] = str(FEATURES_ROOT / fps_dirname(FPS[r["dataset"]]) / r["dataset"] / f"{r['frame_key']}.npz")
                rows.append(r)
    return pd.DataFrame(rows)


rows = load_rows()
rows["has_frames"] = [(Path(d) / "frames.json").exists() for d in rows.frame_dir]
videos = rows.drop_duplicates(["dataset", "frame_key"])
print(f"{len(rows)} questions | {len(videos)} videos | missing frames: {(~videos.has_frames).sum()}")
if (~videos.has_frames).any():
    print("  -> run notebook 01 (it extracts frames for both splits at the per-dataset FPS)")
rows.groupby(["split", "dataset"]).size().rename("questions").to_frame()

## 3. Feature extraction

In [ ]:
from transformers import AutoModel, AutoProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
siglip = AutoModel.from_pretrained(SIGLIP_ID, dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
siglip_proc = AutoProcessor.from_pretrained(SIGLIP_ID)


def _features(out):
    return out if torch.is_tensor(out) else out.pooler_output


def fit_box(im, box=BOX):
    # shrink to fit inside the box, keep the aspect ratio, never upscale
    s = min(1.0, box[0] / im.width, box[1] / im.height)
    return im if s == 1.0 else im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BICUBIC)


@torch.no_grad()
def embed_images(images):
    inp = siglip_proc(images=images, return_tensors="pt").to(DEVICE, siglip.dtype)
    return torch.nn.functional.normalize(_features(siglip.get_image_features(**inp)).float(), dim=-1).cpu().numpy()


@torch.no_grad()
def embed_texts(texts, batch=128):
    out = []
    for i in range(0, len(texts), batch):
        inp = siglip_proc(text=texts[i:i + batch], padding="max_length", max_length=64, truncation=True,
                          return_tensors="pt").to(DEVICE)
        out.append(torch.nn.functional.normalize(_features(siglip.get_text_features(**inp)).float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def segment(emb, fps, min_seg_s, max_seg_s, k_mad):
    n = len(emb)
    d = np.zeros(n, dtype=np.float32)
    if n < 2:
        return np.zeros(n, dtype=np.int32), d
    d[1:] = 1.0 - np.sum(emb[1:] * emb[:-1], axis=1)
    med = np.median(d[1:])
    mad = np.median(np.abs(d[1:] - med)) + 1e-6
    thr = med + k_mad * 1.4826 * mad
    min_len = max(1, int(round(min_seg_s * fps)))
    bounds = []
    for i in np.argsort(-d):
        if d[i] <= thr:
            break
        if i < min_len or n - i < min_len:
            continue
        if all(abs(i - b) >= min_len for b in bounds):
            bounds.append(int(i))
    bounds.sort()
    if max_seg_s:
        max_len = max(1, int(round(max_seg_s * fps)))
        extra = []
        for s, e in zip([0] + bounds, bounds + [n]):
            parts = math.ceil((e - s) / max_len)
            extra += [s + round(j * (e - s) / parts) for j in range(1, parts)]
        bounds = sorted(set(bounds + extra))
    seg = np.zeros(n, dtype=np.int32)
    for b in bounds:
        seg[b:] += 1
    return seg, d


def compute_video(r):
    meta = json.loads((Path(r["frame_dir"]) / "frames.json").read_text())
    frames = meta["frames"]
    fps = FPS[r["dataset"]]
    embs, sharp, contrast, bright, thumbs = [], [], [], [], []
    for i in range(0, len(frames), EMB_BATCH):
        ims = [Image.open(Path(r["frame_dir"]) / f["file"]).convert("RGB") for f in frames[i:i + EMB_BATCH]]
        embs.append(embed_images(ims))
        for im in ims:
            g = cv2.cvtColor(np.asarray(fit_box(im)), cv2.COLOR_RGB2GRAY)
            sharp.append(cv2.Laplacian(g, cv2.CV_64F).var())
            contrast.append(g.std())
            bright.append(g.mean())
            thumbs.append(cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0)
    emb = np.concatenate(embs)
    thumbs = np.stack(thumbs)
    motion = np.zeros(len(frames), dtype=np.float32)
    if len(frames) > 1:
        motion[1:] = np.abs(np.diff(thumbs, axis=0)).mean(axis=(1, 2))
        motion[0] = motion[1]
    seg, d = segment(emb, fps, **SEGMENTATION[r["dataset"]])
    out = Path(r["feature_path"])
    out.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out, version=FEATURE_VERSION, t=np.array([f["t"] for f in frames], dtype=np.float32),
                        emb=emb.astype(np.float16), motion=motion, sharp=np.array(sharp, dtype=np.float32),
                        contrast=np.array(contrast, dtype=np.float32), bright=np.array(bright, dtype=np.float32),
                        seg=seg, boundary=d, fps=fps)
    return len(frames), int(seg.max()) + 1


def is_current(path):
    if not Path(path).exists():
        return False
    with np.load(path) as z:
        return int(z["version"]) == FEATURE_VERSION


todo = [r for r in videos[videos.has_frames].to_dict("records") if not is_current(r["feature_path"])]
print(f"{len(videos) - len(todo)} up to date, {len(todo)} to compute")
failed = []
for r in tqdm(todo, desc="features"):
    try:
        compute_video(r)
    except Exception as e:
        failed.append((r["frame_key"], repr(e)))
print("failed:", len(failed), failed[:5])

In [ ]:
# Question embeddings (question text, and question + options) for every test / calibration question
q_rows = rows.drop_duplicates("qid")
q_text = q_rows.question.tolist()
qo_text = [f"{q} " + " ".join(o) for q, o in zip(q_rows.question, q_rows.options)]
np.savez_compressed(FEATURES_ROOT / "questions.npz", qid=np.array(q_rows.qid.tolist()),
                    q_emb=embed_texts(q_text).astype(np.float16), qo_emb=embed_texts(qo_text).astype(np.float16),
                    model=SIGLIP_ID)
print(len(q_rows), "questions embedded ->", FEATURES_ROOT / "questions.npz")

## 4. Segmentation statistics

In [ ]:
stats = []
for r in videos[videos.has_frames].to_dict("records"):
    if not Path(r["feature_path"]).exists():
        continue
    with np.load(r["feature_path"]) as z:
        n, n_seg = len(z["t"]), int(z["seg"].max()) + 1
        span = float(z["t"][-1] - z["t"][0]) + 1.0 / float(z["fps"])
    stats.append(dict(split=r["split"], dataset=r["dataset"], category=r["category"], frames=n, seconds=span,
                      segments=n_seg, seg_len_s=span / n_seg))
stats = pd.DataFrame(stats)
stats.groupby(["dataset", "category"])[["frames", "seconds", "segments", "seg_len_s"]].mean().round(1)

In [ ]:
# One video: boundary signal, threshold, detected segments and a representative frame per segment
import matplotlib.pyplot as plt

r = videos[videos.has_frames & (videos.dataset == "videomme")].sample(1, random_state=None).iloc[0].to_dict()
z = dict(np.load(r["feature_path"]))
cfg = SEGMENTATION[r["dataset"]]
d = z["boundary"]
thr = np.median(d[1:]) + cfg["k_mad"] * 1.4826 * np.median(np.abs(d[1:] - np.median(d[1:])))
starts = np.r_[0, np.flatnonzero(np.diff(z["seg"])) + 1]

fig, ax = plt.subplots(figsize=(16, 3))
ax.plot(z["t"], d, lw=0.8, label="1 - cos(t, t-1)")
ax.plot(z["t"], z["motion"] / (z["motion"].max() + 1e-8) * d.max(), lw=0.6, alpha=0.6, label="motion (scaled)")
ax.axhline(thr, color="r", ls="--", lw=0.8, label="threshold")
for s in starts:
    ax.axvline(z["t"][s], color="k", lw=0.4, alpha=0.5)
ax.set_xlabel("seconds")
ax.set_title(f"{r['frame_key']} - {len(starts)} segments | Q: {' '.join(r['question'].split())[:120]}")
ax.legend(loc="upper right", fontsize=8)
plt.show()

meta = json.loads((Path(r["frame_dir"]) / "frames.json").read_text())
show = starts[:: max(1, len(starts) // 12)][:12]
fig, axes = plt.subplots(1, len(show), figsize=(2.2 * len(show), 1.8))
for ax, s in zip(np.atleast_1d(axes), show):
    ax.imshow(Image.open(Path(r["frame_dir"]) / meta["frames"][s]["file"]))
    ax.set_title(f"seg {z['seg'][s]} @ {z['t'][s]:.0f}s", fontsize=8)
    ax.axis("off")
plt.show()